# Task 4 - Extended Data Validation Framework

This notebook extends the Week 1 validation (`apply_quality_checks` in `src/common/data_model.py`) into a
rule-based framework that detects

| Requirement | Rule types (category) |
|---|---|
| duplicate records | `unique` (within a file), `not_in_table` (already in Silver) - *duplicate* |
| invalid attribute values | `cast_errors`, `valid_range`, `allowed_values`, `regex`, `expression` - *invalid_value* |
| missing reference records | `reference` (foreign key → lookup table) - *missing_reference* |
| unexpected / unsupported schema changes | `check_source_schema`, `align_types` - *schema* |
| incomplete records | `not_null` - *incomplete* |

Rules are declared per dataset in **`config/datasets-expanded.yaml`**; the engine lives in this notebook and
reuses the Week 1 building blocks (loader, column standardisation, type casting, transformations, Delta writer).

Invalid records **do not interrupt processing** (row-level problems never raise, file-level problems reject only
that file), are **isolated** in the quarantine Delta table, are **reported** in `ingestion_runs` and a new
`validation_reports` Delta table, and are **excluded from analytical data products** because Gold is built from Silver only.

Sections
1. Setup
2. Rule registry and rule types
3. Schema-change detection
4. Validation engine
5. Validated ingestion pipeline
6. Audit of the existing Silver tables (read-only)
7. Test files with injected errors → run in a sandbox
8. Reporting (Spark SQL) and exclusion check
9. Adding a new rule without touching the engine
10. Apply to the Task 1 update files (writes to the platform)
11. Discussion

## 1. Setup

In [1]:
import os
import sys
import json
import shutil
import datetime
from dataclasses import dataclass, field, asdict
from typing import Callable, Dict, List, Optional, Tuple

import yaml
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from deltalake import DeltaTable, write_deltalake
from IPython.display import display


def find_project_root(marker="config/datasets.yaml"):
    path = os.getcwd()
    while path != os.path.dirname(path):
        if os.path.exists(os.path.join(path, marker)):
            return path
        path = os.path.dirname(path)
    raise FileNotFoundError(f"Could not find project root (looking for {marker})")


PROJECT_ROOT = find_project_root()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark import StorageLevel

# Week 1 building blocks, reused unchanged
from src.common.spark_session import get_spark_session
from src.common.config_loader import load_dataset_configs
from src.common.data_model import (
    CAST_ERRORS, IS_VALID, REJECTION_REASON, QUARANTINED_AT,
    add_lineage_columns, change_datatype, normalize_missing_values, standardize_columns,
)
from src.common.delta_writer import write_delta
from src.ingestion.loader import load_source_file
from src.ingestion.metadata import IngestionRunMetadata, RunTimer, new_run_id
from src.monitoring.storage_utils import get_dir_size_bytes
from src.transformation import get_transform

spark = get_spark_session()
spark.sparkContext.setLogLevel("ERROR")
# Parquet files written by deltalake store timestamps without the UTC-adjusted flag; Spark would
# otherwise read them back as timestamp_ntz and every file would look like a type change.
# The session time zone is UTC, so reading them as timestamp does not change any value.
spark.conf.set("spark.sql.parquet.inferTimestampNTZ.enabled", "false")

CONFIG_PATH = os.path.join(PROJECT_ROOT, "config", "datasets-expanded.yaml")
CONFIGS = load_dataset_configs(CONFIG_PATH)          # Week 1 DatasetConfig objects
with open(CONFIG_PATH) as f:
    SPECS = yaml.safe_load(f)["datasets"]            # + the new schema / validation sections

print("Project root:", PROJECT_ROOT)
print("Datasets:", list(CONFIGS))

26/09/26 19:39:21 WARN Utils: Your hostname, DESKTOP-821NIV4 resolves to a loopback address: 127.0.1.1; using 10.10.10.40 instead (on interface wlo1)
26/09/26 19:39:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/26 19:39:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Project root: /home/bogdan/repos/ID2221/LAB
Datasets: ['tripdata', 'weather', 'air_quality', 'taxi_zone_lookup']


In [2]:
@dataclass(frozen=True)
class LayerPaths:
    """Where a pipeline run writes. The platform and the test sandbox use the same layout."""
    root: str

    def table(self, layer: str, dataset: str) -> str:
        return os.path.join(self.root, layer, dataset)

    @property
    def ingestion_runs(self) -> str:
        return os.path.join(self.root, "metadata", "ingestion_runs")

    @property
    def validation_reports(self) -> str:
        return os.path.join(self.root, "metadata", "validation_reports")


PLATFORM = LayerPaths(os.path.join(PROJECT_ROOT, "data", "delta"))
SANDBOX = LayerPaths(os.path.join(PROJECT_ROOT, "data", "delta", "_validation_sandbox"))


def read_delta_spark(path: str) -> Optional[DataFrame]:
    """Read the *current* version of a Delta table with Spark.

    Spark is not configured with delta-spark, so we ask the Delta log (deltalake) for the
    active files and read only those. Reading the directory with spark.read.parquet would
    also pick up files removed by earlier overwrites.
    """
    if not DeltaTable.is_deltatable(path):
        return None
    files = DeltaTable(path).file_uris()
    if not files:
        return None
    return spark.read.option("basePath", path).parquet(*files)

## 2. Rule registry and rule types

A rule type is a function `(df, rule, ctx) -> (df, [(violation_condition, reason), ...])`.
It may add helper columns (prefixed `_v_`, dropped afterwards) - needed for rules that join
other tables. Registering a function under a name is all it takes to make the type usable from YAML.

In [3]:
@dataclass
class RuleType:
    fn: Callable
    category: str
    required_params: Tuple[str, ...] = ()


RULE_REGISTRY: Dict[str, RuleType] = {}
CATEGORIES = ("duplicate", "invalid_value", "missing_reference", "incomplete", "schema")
ACTIONS = ("quarantine", "ignore", "warn")


def register_rule(rule_type: str, category: str, required_params: Tuple[str, ...] = ()):
    assert category in CATEGORIES, category
    def decorator(fn):
        RULE_REGISTRY[rule_type] = RuleType(fn, category, tuple(required_params))
        return fn
    return decorator


@dataclass
class ValidationContext:
    cfg: object                 # Week 1 DatasetConfig
    reference_root: str         # reference tables (Silver lookups, existing keys) are read from here


def _columns(rule: dict, ctx: ValidationContext, key: str = "columns") -> List[str]:
    cols = rule.get(key, [])
    if cols == "$primary_key":
        return list(ctx.cfg.primary_key)
    return [cols] if isinstance(cols, str) else list(cols)


def _nonfinite(col):
    return F.isnan(col.cast("double")) | (F.abs(col.cast("double")) == F.lit(float("inf")))


# ---------------------------------------------------------------- generic rule types

@register_rule("not_null", category="incomplete", required_params=("columns",))
def rule_not_null(df, rule, ctx):
    return df, [(F.col(c).isNull(), f"{rule['name']}:{c}") for c in _columns(rule, ctx)]


@register_rule("cast_errors", category="invalid_value")
def rule_cast_errors(df, rule, ctx):
    """Values that could not be converted to the common data model type (Week 1 change_datatype)."""
    if CAST_ERRORS not in df.columns:
        return df, []
    failed_columns = F.transform(F.col(CAST_ERRORS), lambda e: F.substring_index(e, ":", -1))
    return df, [
        (F.array_contains(failed_columns, c), f"{rule['name']}:{c}")
        for c in df.columns if not c.startswith("_")
    ]


@register_rule("valid_range", category="invalid_value", required_params=("column",))
def rule_valid_range(df, rule, ctx):
    value = F.col(rule["column"])
    bad = _nonfinite(value)
    if rule.get("min") is not None:
        bad = bad | (value < F.lit(rule["min"]))
    if rule.get("max") is not None:
        bad = bad | (value > F.lit(rule["max"]))
    return df, [(value.isNotNull() & bad, rule["name"])]


@register_rule("allowed_values", category="invalid_value", required_params=("column", "values"))
def rule_allowed_values(df, rule, ctx):
    value = F.col(rule["column"])
    return df, [(value.isNotNull() & ~value.isin(list(rule["values"])), rule["name"])]


@register_rule("regex", category="invalid_value", required_params=("column", "pattern"))
def rule_regex(df, rule, ctx):
    value = F.col(rule["column"]).cast("string")
    return df, [(value.isNotNull() & ~value.rlike(rule["pattern"]), rule["name"])]


@register_rule("expression", category="invalid_value", required_params=("valid_if",))
def rule_expression(df, rule, ctx):
    """Any Spark SQL predicate. A NULL result (missing inputs) is left to the not_null rules."""
    return df, [(F.coalesce(~F.expr(rule["valid_if"]), F.lit(False)), rule["name"])]


@register_rule("unique", category="duplicate", required_params=("columns",))
def rule_unique(df, rule, ctx):
    """Duplicates inside the incoming batch. keep=first keeps one copy, keep=none rejects all copies."""
    cols = _columns(rule, ctx)
    complete_key = F.lit(True)
    for c in cols:
        complete_key = complete_key & F.col(c).isNotNull()
    window = Window.partitionBy(*cols)
    if rule.get("keep", "first") == "none":
        bad = F.count(F.lit(1)).over(window) > 1
    else:
        bad = F.row_number().over(window.orderBy(F.col("_v_seq"))) > 1
    return df, [(complete_key & bad, rule["name"])]


@register_rule("not_in_table", category="duplicate", required_params=("columns", "table"))
def rule_not_in_table(df, rule, ctx):
    """Records whose key already exists in a table (e.g. re-delivered rows already in Silver)."""
    cols = _columns(rule, ctx)
    existing = read_delta_spark(os.path.join(ctx.reference_root, rule["table"]))
    if existing is None:
        return df, []
    marker = f"_v_{rule['name']}"
    keys = existing.select(*cols).dropDuplicates().withColumn(marker, F.lit(True))
    df = df.join(keys, on=cols, how="left")
    return df, [(F.col(marker).isNotNull(), rule["name"])]


@register_rule("reference", category="missing_reference", required_params=("column", "table", "ref_column"))
def rule_reference(df, rule, ctx):
    """Foreign key check against a (small) lookup table, broadcast to every executor."""
    ref = read_delta_spark(os.path.join(ctx.reference_root, rule["table"]))
    if ref is None:
        raise ValueError(f"{rule['name']}: reference table '{rule['table']}' does not exist")
    marker = f"_v_{rule['name']}"
    keys = (ref.filter(F.col(IS_VALID)) if IS_VALID in ref.columns else ref) \
        .select(F.col(rule["ref_column"]).alias(marker)).dropDuplicates()
    df = df.join(F.broadcast(keys), F.col(rule["column"]) == F.col(marker), "left")
    return df, [(F.col(rule["column"]).isNotNull() & F.col(marker).isNull(), rule["name"])]


print("Registered rule types:", {k: v.category for k, v in RULE_REGISTRY.items()})

Registered rule types: {'not_null': 'incomplete', 'cast_errors': 'invalid_value', 'valid_range': 'invalid_value', 'allowed_values': 'invalid_value', 'regex': 'invalid_value', 'expression': 'invalid_value', 'unique': 'duplicate', 'not_in_table': 'duplicate', 'reference': 'missing_reference'}


## 3. Schema-change detection

Two checks, run before any row-level rule:

* **`check_source_schema`** - compares the source columns with the configuration:
  * a *required* column (`expected_columns`) is missing → **unsupported**, the file is rejected;
  * a *declared* evolution column (`schema.evolution_columns`, e.g. `humidity`, `aqi`) → **supported**,
    merged into Bronze/Silver automatically;
  * an *undeclared* column → **unexpected**, handled by `schema.on_unexpected_column` (`drop` / `allow` / `reject`).
* **`align_types`** - after standardisation, compares column types with the target Silver table:
  numeric → numeric changes are cast to the Silver type (rows that lose information are flagged as
  `cast_errors`); any other type change (e.g. `double` → `string`) is **unsupported** and rejects the file.

In [4]:
NUMERIC_TYPES = (T.ByteType, T.ShortType, T.IntegerType, T.LongType, T.FloatType, T.DoubleType, T.DecimalType)


@dataclass
class SchemaReport:
    status: str = "OK"                                   # OK | EVOLVED | REJECTED
    missing_required: List[str] = field(default_factory=list)
    evolved: List[str] = field(default_factory=list)     # declared, new to the target table
    unexpected: List[str] = field(default_factory=list)  # undeclared
    unexpected_action: str = ""
    casts: List[str] = field(default_factory=list)       # compatible numeric type changes
    incompatible: List[str] = field(default_factory=list)

    def reject(self):
        self.status = "REJECTED"

    @property
    def message(self) -> str:
        parts = []
        if self.missing_required:
            parts.append(f"missing required columns {self.missing_required}")
        if self.incompatible:
            parts.append(f"incompatible type changes {self.incompatible}")
        if self.unexpected:
            parts.append(f"unexpected columns {self.unexpected} ({self.unexpected_action})")
        if self.evolved:
            parts.append(f"schema evolution {self.evolved}")
        if self.casts:
            parts.append(f"numeric casts {self.casts}")
        return "; ".join(parts)

    def as_json(self) -> str:
        return json.dumps({k: v for k, v in asdict(self).items() if v and k != "status"})


def check_source_schema(raw_df: DataFrame, cfg, spec: dict, target_columns: set) -> SchemaReport:
    policy = spec.get("schema", {})
    report = SchemaReport()
    required = list(cfg.expected_columns)
    known = set(required) | set(policy.get("known_columns", []))
    declared = set(policy.get("evolution_columns", []))

    report.missing_required = [c for c in required if c not in raw_df.columns]
    extra = [c for c in raw_df.columns if c not in known]
    report.unexpected = [c for c in extra if c not in declared]
    # a declared evolution column counts as a schema change only the first time it arrives
    report.evolved = [c for c in extra if c in declared
                      and cfg.column_mapping.get(c, c) not in target_columns]

    if report.missing_required:
        report.reject()
    if report.unexpected:
        report.unexpected_action = policy.get("on_unexpected_column", "drop")
        if report.unexpected_action == "reject":
            report.reject()
    if report.status != "REJECTED" and (report.evolved or (report.unexpected and report.unexpected_action == "allow")):
        report.status = "EVOLVED"
    return report


def align_types(df: DataFrame, target_schema: Optional[T.StructType], report: SchemaReport) -> DataFrame:
    if target_schema is None:
        return df
    target = {f.name: f.dataType for f in target_schema.fields}
    if CAST_ERRORS not in df.columns:
        df = df.withColumn(CAST_ERRORS, F.array().cast("array<string>"))
    for f in df.schema.fields:
        if f.name.startswith("_") or f.name not in target or f.dataType == target[f.name]:
            continue
        wanted = target[f.name]
        if isinstance(f.dataType, NUMERIC_TYPES) and isinstance(wanted, NUMERIC_TYPES):
            report.casts.append(f"{f.name}: {f.dataType.simpleString()} -> {wanted.simpleString()}")
            converted = F.col(f.name).cast(wanted)
            lossy = F.col(f.name).isNotNull() & (converted.isNull() | (converted.cast("double") != F.col(f.name).cast("double")))
            df = df.withColumn(CAST_ERRORS, F.concat(F.col(CAST_ERRORS), F.array(F.when(lossy, F.lit(f"invalid_numeric:{f.name}")))))
            df = df.withColumn(f.name, F.when(lossy, F.lit(None).cast(wanted)).otherwise(converted))
        else:
            report.incompatible.append(f"{f.name}: {wanted.simpleString()} -> {f.dataType.simpleString()}")
    if report.incompatible:
        report.reject()
    return df.withColumn(CAST_ERRORS, F.filter(F.col(CAST_ERRORS), lambda x: x.isNotNull()))

## 4. Validation engine

`ExtendedValidator` knows nothing about individual datasets or rules: it reads the rule list, looks each
rule type up in the registry and collects the violation reasons of every row into three arrays, one per action:

* `quarantine` → row is isolated in the quarantine table (`rejection_reason` lists **all** violated rules);
* `ignore` → row is dropped and counted (re-delivered records that are already in Silver);
* `warn` → row stays in Silver, the violation is only reported.

In [5]:
HELPER_PREFIX = "_v_"


class ExtendedValidator:
    def __init__(self, cfg, spec: dict, reference_root: str, extra_rules=(), skip_types=()):
        self.ctx = ValidationContext(cfg=cfg, reference_root=reference_root)
        rules = list(spec.get("validation", {}).get("rules", [])) + list(extra_rules)
        self.rules = [r for r in rules if r["type"] not in skip_types]
        self._check_config()
        declared = spec.get("schema", {}).get("evolution_columns", [])
        self.pending_columns = {cfg.column_mapping.get(c, c) for c in declared}
        self.skipped = []
        # reason prefix (the rule name) -> (category, action), used for reporting
        self.rule_index = {
            r["name"]: (r.get("category", RULE_REGISTRY[r["type"]].category), r.get("action", "quarantine"))
            for r in self.rules
        }

    def _check_config(self):
        """Fail fast on a broken configuration instead of failing half-way through the data."""
        seen = set()
        for r in self.rules:
            name = r.get("name")
            if not name or name in seen:
                raise ValueError(f"{self.ctx.cfg.name}: every rule needs a unique name ({r})")
            seen.add(name)
            if r.get("type") not in RULE_REGISTRY:
                raise ValueError(f"{self.ctx.cfg.name}.{name}: unknown rule type '{r.get('type')}'. "
                                 f"Registered: {sorted(RULE_REGISTRY)}")
            missing = [p for p in RULE_REGISTRY[r["type"]].required_params if p not in r]
            if missing:
                raise ValueError(f"{self.ctx.cfg.name}.{name}: missing parameters {missing}")
            if r.get("action", "quarantine") not in ACTIONS:
                raise ValueError(f"{self.ctx.cfg.name}.{name}: action must be one of {ACTIONS}")

    def _referenced_columns(self, rule: dict) -> List[str]:
        cols = _columns(rule, self.ctx) + [rule[k] for k in ("column", "numerator", "denominator") if k in rule]
        return cols

    def validate(self, df: DataFrame) -> DataFrame:
        """Adds _v_quarantine / _v_ignore / _v_warn (array<string>) and _v_status."""
        df = df.withColumn("_v_seq", F.monotonically_increasing_id())
        by_action = {a: [] for a in ACTIONS}
        self.skipped = []
        for rule in self.rules:
            absent = [c for c in self._referenced_columns(rule) if c not in df.columns]
            if absent:
                # a declared evolution column that has not arrived yet (e.g. humidity in old data)
                if set(absent) <= self.pending_columns:
                    self.skipped.append(rule["name"])
                    continue
                raise ValueError(f"{self.ctx.cfg.name}.{rule['name']}: columns {absent} do not exist")
            df, violations = RULE_REGISTRY[rule["type"]].fn(df, rule, self.ctx)
            for condition, reason in violations:
                by_action[rule.get("action", "quarantine")].append(
                    F.when(F.coalesce(condition, F.lit(False)), F.lit(reason)))

        for action, reasons in by_action.items():
            arr = F.array(*reasons) if reasons else F.array().cast("array<string>")
            df = df.withColumn(f"_v_{action}", F.filter(arr, lambda x: x.isNotNull()))

        return df.withColumn(
            "_v_status",
            F.when(F.size("_v_ignore") > 0, "ignored")
             .when(F.size("_v_quarantine") > 0, "quarantined")
             .otherwise("valid"),
        )

    def reason_counts(self, validated: DataFrame) -> pd.DataFrame:
        """One row per (reason, action) with the number of violating records."""
        def tagged(action):
            return F.transform(F.col(f"_v_{action}"),
                               lambda r: F.struct(r.alias("reason"), F.lit(action).alias("action")))
        events = F.concat(*[tagged(a) for a in ACTIONS])
        pdf = (validated.select(F.explode(events).alias("e"))
               .groupBy("e.reason", "e.action").count()
               .toPandas().rename(columns={"count": "violations"}))
        pdf["rule_name"] = pdf["reason"].str.split(":").str[0]
        pdf["category"] = pdf["rule_name"].map(lambda n: self.rule_index.get(n, ("invalid_value",))[0])
        return pdf.sort_values("violations", ascending=False, ignore_index=True)

    @staticmethod
    def strip_helpers(df: DataFrame) -> DataFrame:
        return df.drop(*[c for c in df.columns if c.startswith(HELPER_PREFIX) or c == CAST_ERRORS])


for name, spec in SPECS.items():
    v = ExtendedValidator(CONFIGS[name], spec, PLATFORM.root)
    print(f"{name:18s} {len(v.rules):2d} rules:", ", ".join(r["name"] for r in v.rules))

tripdata           16 rules: type_conformance, incomplete, duplicate_in_batch, duplicate_existing, invalid_trip_distance, invalid_passenger_count, invalid_pickup_year, invalid_pickup_month, invalid_vendor_id, invalid_payment_type, invalid_ratecode_id, dropoff_before_pickup, trip_longer_than_24h, missing_pickup_zone, missing_dropoff_zone, negative_fare
weather            12 rules: type_conformance, incomplete, duplicate_in_batch, duplicate_existing, invalid_temperature, invalid_relative_humidity, invalid_humidity, invalid_precipitation, invalid_wind_speed, invalid_wind_direction, invalid_pressure, invalid_condition_code
air_quality        10 rules: type_conformance, incomplete, duplicate_in_batch, duplicate_existing, invalid_latitude, invalid_longitude, invalid_pm25_concentration, invalid_aqi, unexpected_parameter_code, invalid_time_gmt
taxi_zone_lookup    4 rules: type_conformance, incomplete, duplicate_in_batch, invalid_location_id


## 5. Validated ingestion pipeline

Same stages as the Week 1 ingestor (`src/ingestion/ingestor.py`), with the new checks in between:

`load → schema check → standardise/cast/transform (Week 1) → type alignment → validate → Bronze / Silver / Quarantine → report`

Every run appends one row to `metadata/ingestion_runs` (the table used by the Task 3 monitoring queries, with
extra columns `pipeline_name`, `schema_status`, `schema_changes`, `records_warned`) and one row per violated rule
to `metadata/validation_reports`. Exceptions are caught and recorded, so one bad file never stops the others.

In [7]:
PIPELINE_NAME = "validated_ingestion"


def _write_run_reports(paths: LayerPaths, metadata: dict, reasons: pd.DataFrame):
    write_deltalake(paths.ingestion_runs, pd.DataFrame([metadata]), mode="append", schema_mode="merge")
    if reasons is not None and not reasons.empty:
        rows = reasons.assign(
            run_id=metadata["run_id"], dataset_name=metadata["dataset_name"],
            source_file=metadata["source_file"], reported_at=metadata["ended_at"],
        )[["run_id", "dataset_name", "source_file", "rule_name", "reason", "category", "action", "violations", "reported_at"]]
        rows["violations"] = rows["violations"].astype("int64")
        write_deltalake(paths.validation_reports, rows, mode="append", schema_mode="merge")


def _schema_reasons(report: SchemaReport, n_rows: int) -> pd.DataFrame:
    rows = [(f"missing_required_column:{c}", "reject_file") for c in report.missing_required]
    rows += [(f"incompatible_type:{c.split(':')[0]}", "reject_file") for c in report.incompatible]
    rows += [(f"unexpected_column:{c}", report.unexpected_action if report.unexpected_action != "reject" else "reject_file")
             for c in report.unexpected]
    rows += [(f"schema_evolution:{c}", "evolve") for c in report.evolved]
    rows += [(f"numeric_cast:{c.split(':')[0]}", "cast") for c in report.casts]
    pdf = pd.DataFrame(rows, columns=["reason", "action"])
    pdf["violations"] = n_rows
    pdf["rule_name"] = "schema"
    pdf["category"] = "schema"
    return pdf


def run_validated_ingestion(dataset: str, source_path: str, paths: LayerPaths = PLATFORM,
                            reference: Optional[LayerPaths] = None, extra_rules=(),
                            dry_run: bool = False, record: bool = True) -> dict:
    """paths: where to write. reference: where existing keys / lookup tables are read (default: platform).
    dry_run: validate only, write nothing. record=False: write the data but skip monitoring
    (run metadata, validation reports, storage measurement) - used to measure monitoring overhead."""
    cfg, spec = CONFIGS[dataset], SPECS[dataset]
    reference = reference or PLATFORM
    record = record and not dry_run
    run_id, timer = new_run_id(), RunTimer()
    meta = asdict(IngestionRunMetadata(run_id=run_id, dataset_name=dataset, source_file=source_path,
                                       format=cfg.format, schema_version=cfg.schema_version, status="FAILED"))
    meta.update(pipeline_name=PIPELINE_NAME, schema_status="", schema_changes="{}", records_warned=0)
    reasons = pd.DataFrame()
    validated = None
    silver_path = paths.table("silver", dataset)
    silver_before = get_dir_size_bytes(silver_path) if record else 0

    with timer:
        try:
            raw_df = load_source_file(spark, cfg, source_path)
            if cfg.filter_expr:
                raw_df = raw_df.filter(cfg.filter_expr)

            # the target Silver table; an empty target falls back to the reference schema
            target = read_delta_spark(silver_path)
            if target is None:
                target = read_delta_spark(reference.table("silver", dataset))
            target_schema = target.schema if target is not None else None
            target_columns = set(target.columns) if target is not None else set()

            # 1. source schema: missing / declared / unexpected columns
            schema = check_source_schema(raw_df, cfg, spec, target_columns)
            if schema.status != "REJECTED":
                silver_input = raw_df.drop(*schema.unexpected) if schema.unexpected_action == "drop" else raw_df

                # 2. Week 1 standardisation, type normalisation and dataset transformation
                df = standardize_columns(silver_input, cfg.column_mapping)
                df = normalize_missing_values(df)
                df = change_datatype(df, cfg.dtype_overrides)
                df = get_transform(cfg.transform_module)(df, spark)
                df = add_lineage_columns(df, source_path, cfg.schema_version)

                # 3. column types against the target table
                df = align_types(df, target_schema, schema)

            meta["schema_status"] = schema.status
            meta["schema_changes"] = schema.as_json()
            records_read = raw_df.count()
            meta["records_read"] = records_read

            if schema.status == "REJECTED":
                # file-level isolation: nothing is written to Bronze/Silver, the file is reported
                meta.update(status="SCHEMA_REJECTED", error_message=schema.message)
                reasons = _schema_reasons(schema, records_read)
            else:
                # 4. row-level validation
                validator = ExtendedValidator(cfg, spec, reference.root, extra_rules=extra_rules)
                validated = validator.validate(df).persist(StorageLevel.MEMORY_AND_DISK)
                status_counts = {r["_v_status"]: r["count"] for r in validated.groupBy("_v_status").count().collect()}
                reasons = pd.concat([validator.reason_counts(validated), _schema_reasons(schema, records_read)],
                                    ignore_index=True)

                valid_df = (ExtendedValidator.strip_helpers(validated.filter(F.col("_v_status") == "valid"))
                            .withColumn(REJECTION_REASON, F.lit(None).cast("string"))
                            .withColumn(IS_VALID, F.lit(True)))
                quarantine_df = (ExtendedValidator.strip_helpers(
                                    validated.filter(F.col("_v_status") == "quarantined")
                                             .withColumn(REJECTION_REASON, F.array_join("_v_quarantine", ", ")))
                                 .withColumn(IS_VALID, F.lit(False))
                                 .withColumn(QUARANTINED_AT, F.current_timestamp())
                                 .withColumn("run_id", F.lit(run_id)))

                if not dry_run:
                    write_delta(add_lineage_columns(raw_df, source_path, cfg.schema_version),
                                paths.table("bronze", dataset), mode="append", schema_mode="merge")
                    write_delta(valid_df, silver_path, mode=cfg.write_mode,
                                partition_by=cfg.partition_by, schema_mode="merge")
                    if status_counts.get("quarantined", 0):
                        write_delta(quarantine_df, paths.table("quarantine", dataset), mode="append", schema_mode="merge")

                failures = reasons[reasons["action"] == "quarantine"]
                meta.update(
                    status="SUCCESS" if not dry_run else "DRY_RUN",
                    records_bronze=0 if dry_run else records_read,
                    records_valid_silver=status_counts.get("valid", 0),
                    records_quarantined=status_counts.get("quarantined", 0),
                    records_duplicate_ignored=status_counts.get("ignored", 0),
                    records_warned=int(validated.filter((F.col("_v_status") == "valid") & (F.size("_v_warn") > 0)).count()),
                    validation_failures=json.dumps({r: int(n) for r, n in zip(failures["reason"], failures["violations"])}),
                )
        except Exception as exc:  # noqa: BLE001 - a failing file must not stop the other files
            meta.update(status="FAILED", error_message=f"{type(exc).__name__}: {exc}"[:2000])
        finally:
            if validated is not None:
                validated.unpersist()

    meta.update(started_at=timer.started_at.isoformat(), ended_at=timer.ended_at.isoformat(),
                execution_time_seconds=round(timer.elapsed_seconds, 3))
    if record:
        meta["storage_overhead_bytes"] = get_dir_size_bytes(silver_path) - silver_before
        _write_run_reports(paths, meta, reasons)
    meta["reasons"] = reasons
    return meta


SUMMARY_COLUMNS = ["dataset_name", "source_file", "status", "schema_status", "records_read",
                   "records_valid_silver", "records_quarantined", "records_duplicate_ignored",
                   "records_warned", "execution_time_seconds", "error_message"]


def summarize(results: List[dict]) -> pd.DataFrame:
    pdf = pd.DataFrame([{k: r[k] for k in SUMMARY_COLUMNS} for r in results])
    pdf["source_file"] = pdf["source_file"].map(os.path.basename)
    return pdf

## 6. Audit of the existing Silver tables (read-only)

The extended rules are stricter than the Week 1 checks. Before using them on new data we run them over the
records already in Silver (`duplicate_existing` is skipped - every Silver row is, by definition, in Silver).
Nothing is written: this shows how much of the current platform the new rules would reject.

In [8]:
audit_rows = []
for name in CONFIGS:
    silver = read_delta_spark(PLATFORM.table("silver", name))
    if silver is None:
        print(f"{name}: no Silver table, skipped")
        continue
    validator = ExtendedValidator(CONFIGS[name], SPECS[name], PLATFORM.root, skip_types=("not_in_table",))
    validated = validator.validate(silver).persist(StorageLevel.MEMORY_AND_DISK)
    total = validated.count()
    if validator.skipped:
        print(f"{name}: skipped {validator.skipped} (evolution columns not in the table yet)")
    for _, r in validator.reason_counts(validated).iterrows():
        audit_rows.append({"dataset": name, "silver_rows": total, "reason": r["reason"], "category": r["category"],
                           "action": r["action"], "violations": r["violations"],
                           "pct": round(100 * r["violations"] / total, 4)})
    validated.unpersist()

audit = pd.DataFrame(audit_rows)
display(audit if not audit.empty else "No violations in the existing Silver tables.")

weather: skipped ['invalid_humidity'] (evolution columns not in the table yet)


,dataset,silver_rows,reason,category,action,violations,pct
0,tripdata,9848446,incomplete:dropoff_timestamp,incomplete,quarantine,269406,2.7355
1,tripdata,9848446,negative_fare,invalid_value,warn,141392,1.4357
2,tripdata,9848446,dropoff_before_pickup,invalid_value,quarantine,12264,0.1245
3,tripdata,9848446,trip_longer_than_24h,invalid_value,quarantine,10590,0.1075
4,tripdata,9848446,invalid_trip_distance,invalid_value,quarantine,169,0.0017
5,tripdata,9848446,invalid_pickup_year,invalid_value,quarantine,9,0.0001


## 7. Test files with injected errors → sandbox run

The Task 1 update files are (almost) clean, so they cannot show that every detector works. We build small test
files from the real sources and inject known problems. They are ingested into a **sandbox**
(`data/delta/_validation_sandbox`) so the platform tables are not modified; reference data (zone lookup, existing
Silver keys) is still read from the platform.

| File | Injected problems |
|---|---|
| `tripdata_dirty.parquet` | unknown zone 999, negative distance, dropoff before pickup, missing pickup time, missing fare, negative fare (warn), unknown vendor, 30 in-file duplicates, 30 rows already in Silver |
| `tripdata_type_change.parquet` | `congestion_surcharge` delivered as string → unsupported type change |
| `weather_evolved.csv` | new `humidity` (declared) and `uv_index` (undeclared) columns, out-of-range temperature/humidity, missing humidity value, conflicting duplicate hour, rows already in Silver |
| `weather_missing_column.csv` | required column `pres` removed → unsupported schema change |
| `air_quality_evolved.csv` | new `aqi` column, AQI 900, non-numeric measurement, missing latitude, ozone parameter code, invalid GMT time, rows already in Silver |

In [9]:
TEST_DIR = os.path.join(PROJECT_ROOT, "data", "raw", "_validation_tests")   # outside every dataset's raw_path glob
os.makedirs(TEST_DIR, exist_ok=True)
N_INJECT = 20


def inject(pdf, start, column, value):
    pdf.loc[start:start + N_INJECT - 1, column] = value
    return start + N_INJECT


# ---------------------------------------------------------------- tripdata
trip_src = os.path.join(PROJECT_ROOT, "data", "raw", "tripdata", "yellow_tripdata_2024-03.parquet")
trip_table = pq.read_table(trip_src)
trip_schema = trip_table.schema
march = trip_table.slice(0, 200_000).to_pandas()
march = march[(march["tpep_pickup_datetime"] >= "2024-03-01") & (march["tpep_pickup_datetime"] < "2024-03-15")]
march = march[(march["tpep_dropoff_datetime"] > march["tpep_pickup_datetime"]) & (march["fare_amount"] > 0)
              & (march["trip_distance"] >= 0) & march["VendorID"].isin([1, 2])]
march = march[march["passenger_count"].isna() | (march["passenger_count"] <= 9)]

base = march.iloc[:5000].copy().reset_index(drop=True)
shift = pd.Timedelta(days=31)                        # 1-14 March -> 1-14 April: new trips
base["tpep_pickup_datetime"] += shift
base["tpep_dropoff_datetime"] += shift

i = 0
i = inject(base, i, "PULocationID", 999)                                           # missing reference
i = inject(base, i, "trip_distance", -3.0)                                         # invalid value
base.loc[i:i + N_INJECT - 1, "tpep_dropoff_datetime"] = base.loc[i:i + N_INJECT - 1, "tpep_pickup_datetime"] - pd.Timedelta(minutes=10)
i += N_INJECT                                                                      # invalid value (dropoff < pickup)
i = inject(base, i, "tpep_pickup_datetime", pd.NaT)                                # incomplete
i = inject(base, i, "fare_amount", float("nan"))                                   # incomplete
i = inject(base, i, "fare_amount", -5.0)                                           # warning only
i = inject(base, i, "VendorID", 3)                                                 # invalid categorical value

in_file_dups = base.iloc[200:230]                                                  # duplicates inside the file
already_in_silver = march.iloc[10_000:10_030]                                      # unchanged March rows
tripdata_dirty = pd.concat([base, in_file_dups, already_in_silver], ignore_index=True)
TRIP_DIRTY = os.path.join(TEST_DIR, "tripdata_dirty.parquet")
pq.write_table(pa.Table.from_pandas(tripdata_dirty, schema=trip_schema, preserve_index=False), TRIP_DIRTY)

type_change = march.iloc[5000:5500].copy()
type_change["tpep_pickup_datetime"] += pd.Timedelta(days=32)
type_change["tpep_dropoff_datetime"] += pd.Timedelta(days=32)
type_change["congestion_surcharge"] = type_change["congestion_surcharge"].map(lambda v: f"{v:.2f} USD")
type_schema = trip_schema.set(trip_schema.get_field_index("congestion_surcharge"), pa.field("congestion_surcharge", pa.string()))
TRIP_TYPE_CHANGE = os.path.join(TEST_DIR, "tripdata_type_change.parquet")
pq.write_table(pa.Table.from_pandas(type_change, schema=type_schema, preserve_index=False), TRIP_TYPE_CHANGE)

# ---------------------------------------------------------------- weather
weather_src = pd.read_csv(os.path.join(PROJECT_ROOT, "data", "raw", "weather", "weather.csv"), dtype=str)
last = weather_src.iloc[-1]
start_ts = pd.Timestamp(int(last.year), int(last.month), int(last.day), int(last.hour))
rows = []
for h in range(1, 49):                                                             # 48 new hours
    ts = start_ts + pd.Timedelta(hours=h)
    template = weather_src.iloc[h].copy()
    template[["year", "month", "day", "hour"]] = [str(ts.year), str(ts.month), str(ts.day), str(ts.hour)]
    rows.append(template)
weather_new = pd.DataFrame(rows).reset_index(drop=True)
weather_new["humidity"] = [f"{v:.1f}" for v in pd.Series(range(48)) * 1.5 + 25]   # declared evolution
weather_new["uv_index"] = "3"                                                      # undeclared column
weather_new.loc[0:1, "temp"] = "99.0"                                              # invalid value
weather_new.loc[2:3, "humidity"] = "140"                                           # invalid value (new column)
weather_new.loc[4, "rhum"] = ""                                                    # incomplete
conflict = weather_new.iloc[[10]].copy()
conflict["temp"] = "-3.0"                                                          # same hour, different reading
already = weather_src.iloc[100:102].copy()                                         # already in Silver
weather_dirty = pd.concat([weather_new, conflict, already], ignore_index=True)
WEATHER_DIRTY = os.path.join(TEST_DIR, "weather_evolved.csv")
weather_dirty.to_csv(WEATHER_DIRTY, index=False)

WEATHER_MISSING = os.path.join(TEST_DIR, "weather_missing_column.csv")
weather_new.drop(columns=["pres", "humidity", "uv_index"]).to_csv(WEATHER_MISSING, index=False)

# ---------------------------------------------------------------- air quality
aq_src = pd.read_csv(os.path.join(PROJECT_ROOT, "data", "raw", "air_quality", "air_quality.csv"), dtype=str, nrows=3000)
aq_new = aq_src.iloc[:2000].copy().reset_index(drop=True)
for date_col, time_col in [("Date Local", "Time Local"), ("Date GMT", "Time GMT")]:
    shifted = pd.to_datetime(aq_new[date_col] + " " + aq_new[time_col]) + pd.Timedelta(days=400)
    aq_new[date_col] = shifted.dt.strftime("%Y-%m-%d")
    aq_new[time_col] = shifted.dt.strftime("%H:%M")
aq_new["aqi"] = [str(v % 150) for v in range(len(aq_new))]                         # declared evolution
i = 0
i = inject(aq_new, i, "aqi", "900")                                                # invalid value (new column)
i = inject(aq_new, i, "Sample Measurement", "n/a")                                 # type conformance
i = inject(aq_new, i, "Latitude", "")                                              # incomplete
i = inject(aq_new, i, "Parameter Code", "44201")                                   # ozone, not PM2.5
i = inject(aq_new, i, "Time GMT", "25:00")                                         # invalid time -> no timestamp
already = aq_src.iloc[2500:2520].copy()                                            # already in Silver
AQ_DIRTY = os.path.join(TEST_DIR, "air_quality_evolved.csv")
pd.concat([aq_new, already], ignore_index=True).to_csv(AQ_DIRTY, index=False)

TEST_FILES = [
    ("tripdata", TRIP_DIRTY), ("tripdata", TRIP_TYPE_CHANGE),
    ("weather", WEATHER_DIRTY), ("weather", WEATHER_MISSING),
    ("air_quality", AQ_DIRTY),
]
for ds, path in TEST_FILES:
    print(f"{ds:12s} {os.path.basename(path)}")

tripdata     tripdata_dirty.parquet
tripdata     tripdata_type_change.parquet
weather      weather_evolved.csv
weather      weather_missing_column.csv
air_quality  air_quality_evolved.csv


In [10]:
shutil.rmtree(SANDBOX.root, ignore_errors=True)       # fresh sandbox on every run

test_results = [run_validated_ingestion(ds, path, paths=SANDBOX) for ds, path in TEST_FILES]
display(summarize(test_results))

,dataset_name,source_file,status,schema_status,records_read,records_valid_silver,records_quarantined,records_duplicate_ignored,records_warned,execution_time_seconds,error_message
0,tripdata,tripdata_dirty.parquet,SUCCESS,OK,5060,4880,150,30,20,5.786,
1,tripdata,tripdata_type_change.parquet,SCHEMA_REJECTED,REJECTED,500,0,0,0,0,0.491,incompatible type changes ['congestion_surchar...
2,weather,weather_evolved.csv,SUCCESS,EVOLVED,51,0,0,51,0,2.914,
3,weather,weather_missing_column.csv,SCHEMA_REJECTED,REJECTED,48,0,0,0,0,0.143,missing required columns ['pres']
4,air_quality,air_quality_evolved.csv,SUCCESS,OK,2020,1900,100,20,0,5.304,


In [11]:
for r in test_results:
    print(f"\n=== {r['dataset_name']} / {os.path.basename(r['source_file'])}: {r['status']}")
    if r["schema_changes"] != "{}":
        print("schema:", r["schema_changes"])
    if not r["reasons"].empty:
        display(r["reasons"][["category", "reason", "action", "violations"]])


=== tripdata / tripdata_dirty.parquet: SUCCESS


,category,reason,action,violations
0,duplicate,duplicate_in_batch,quarantine,30
1,duplicate,duplicate_existing,ignore,30
2,invalid_value,invalid_trip_distance,quarantine,20
3,missing_reference,missing_pickup_zone,quarantine,20
4,invalid_value,dropoff_before_pickup,quarantine,20
5,incomplete,incomplete:pickup_timestamp,quarantine,20
6,invalid_value,negative_fare,warn,20
7,incomplete,incomplete:event_timestamp,quarantine,20
8,invalid_value,invalid_vendor_id,quarantine,20
9,incomplete,incomplete:fare_amount,quarantine,20



=== tripdata / tripdata_type_change.parquet: SCHEMA_REJECTED
schema: {"incompatible": ["congestion_surcharge: double -> string"]}


,category,reason,action,violations
0,schema,incompatible_type:congestion_surcharge,reject_file,500



=== weather / weather_evolved.csv: SUCCESS
schema: {"evolved": ["humidity"], "unexpected": ["uv_index"], "unexpected_action": "drop"}


,category,reason,action,violations
0,duplicate,duplicate_existing,ignore,51
1,invalid_value,invalid_humidity,quarantine,2
2,invalid_value,invalid_temperature,quarantine,2
3,duplicate,duplicate_in_batch,quarantine,2
4,incomplete,incomplete:relative_humidity,quarantine,1
5,schema,unexpected_column:uv_index,drop,51
6,schema,schema_evolution:humidity,evolve,51



=== weather / weather_missing_column.csv: SCHEMA_REJECTED
schema: {"missing_required": ["pres"]}


,category,reason,action,violations
0,schema,missing_required_column:pres,reject_file,48



=== air_quality / air_quality_evolved.csv: SUCCESS


,category,reason,action,violations
0,incomplete,incomplete:pm25_concentration,quarantine,20
1,incomplete,incomplete:latitude,quarantine,20
2,invalid_value,unexpected_parameter_code,quarantine,20
3,duplicate,duplicate_existing,ignore,20
4,invalid_value,invalid_aqi,quarantine,20
5,invalid_value,type_conformance:pm25_concentration,quarantine,20
6,invalid_value,invalid_time_gmt,quarantine,20
7,incomplete,incomplete:event_timestamp,quarantine,20


## 8. Reporting (Spark SQL) and exclusion check

In [12]:
def register_views(paths: LayerPaths, suffix: str = ""):
    for view, path in [("ingestion_runs", paths.ingestion_runs), ("validation_reports", paths.validation_reports)]:
        df = read_delta_spark(path)
        if df is not None:
            df.createOrReplaceTempView(view + suffix)
    for ds in CONFIGS:
        df = read_delta_spark(paths.table("quarantine", ds))
        if df is not None:
            df.createOrReplaceTempView(f"quarantine_{ds}{suffix}")


register_views(SANDBOX, "_sandbox")

print("Violations per dataset and category")
spark.sql("""
    SELECT dataset_name, category, action, SUM(violations) AS violations, COUNT(DISTINCT reason) AS distinct_reasons
    FROM validation_reports_sandbox
    GROUP BY dataset_name, category, action
    ORDER BY dataset_name, violations DESC
""").show(truncate=False)

print("Rejected / isolated records per run")
spark.sql("""
    SELECT dataset_name, regexp_extract(source_file, '[^/]+$', 0) AS file, status, schema_status,
           records_read, records_quarantined, records_duplicate_ignored, records_warned
    FROM ingestion_runs_sandbox
    WHERE pipeline_name = 'validated_ingestion'
    ORDER BY started_at
""").show(truncate=False)

print("Most frequent quarantine reasons (tripdata)")
if spark.catalog.tableExists("quarantine_tripdata_sandbox"):
  spark.sql("""
    SELECT reason, COUNT(*) AS records
    FROM quarantine_tripdata_sandbox
    LATERAL VIEW explode(split(rejection_reason, ', ')) r AS reason
    GROUP BY reason ORDER BY records DESC
  """).show(truncate=False)

Violations per dataset and category
+------------+-----------------+-----------+----------+----------------+
|dataset_name|category         |action     |violations|distinct_reasons|
+------------+-----------------+-----------+----------+----------------+
|air_quality |invalid_value    |quarantine |80        |4               |
|air_quality |incomplete       |quarantine |60        |3               |
|air_quality |duplicate        |ignore     |20        |1               |
|tripdata    |schema           |reject_file|500       |1               |
|tripdata    |invalid_value    |quarantine |60        |3               |
|tripdata    |incomplete       |quarantine |60        |3               |
|tripdata    |duplicate        |quarantine |30        |1               |
|tripdata    |duplicate        |ignore     |30        |1               |
|tripdata    |missing_reference|quarantine |20        |1               |
|tripdata    |invalid_value    |warn       |20        |1               |
|weather     |s

**Exclusion from analytical products.** Gold (`integrated_taxi_trips`) and the Week 2 data products are built from
Silver only, so it is enough to verify that no isolated record reaches Silver - and, for the platform, that no
quarantined trip reached Gold. Extra copies of an in-file duplicate are the exception by design: one copy is kept
in Silver (`keep: first`) and the others are quarantined under the same key, so they are counted separately.

In [12]:
checks = []
for ds in CONFIGS:
    silver = read_delta_spark(SANDBOX.table("silver", ds))
    quarantine = read_delta_spark(SANDBOX.table("quarantine", ds))
    if silver is None or quarantine is None:
        continue
    keys = list(CONFIGS[ds].primary_key)
    # extra copies of an in-file duplicate share the key of the copy kept in Silver (keep: first)
    extra_copies = quarantine.filter(F.col(REJECTION_REASON) == "duplicate_in_batch")
    isolated = quarantine.subtract(extra_copies)
    leaked = isolated.select(*keys).dropna().join(silver.select(*keys), on=keys, how="inner").count()
    checks.append({"scope": "sandbox", "dataset": ds, "silver_rows": silver.count(),
                   "quarantined_rows": quarantine.count(), "duplicate_copies": extra_copies.count(),
                   "quarantined_keys_in_silver": leaked})

gold = read_delta_spark(os.path.join(PLATFORM.root, "gold", "integrated_taxi_trips"))
platform_q = read_delta_spark(PLATFORM.table("quarantine", "tripdata"))
if gold is not None and platform_q is not None:
    leaked = platform_q.select("trip_id").join(gold.select("trip_id"), "trip_id").count()
    checks.append({"scope": "platform", "dataset": "gold.integrated_taxi_trips", "silver_rows": None,
                   "quarantined_rows": platform_q.count(), "quarantined_keys_in_silver": leaked})

display(pd.DataFrame(checks))

,scope,dataset,silver_rows,quarantined_rows,duplicate_copies,quarantined_keys_in_silver
0,sandbox,tripdata,4880.0,150,30.0,0
1,sandbox,weather,42.0,7,2.0,0
2,sandbox,air_quality,1900.0,100,0.0,0
3,platform,gold.integrated_taxi_trips,NaN,2,NaN,0


## 9. Adding a new rule without touching the engine

* **New rule of an existing type** - add an entry to `validation.rules` in `config/datasets-expanded.yaml`
  (e.g. another `valid_range` or an `expression` with any Spark SQL predicate). No code change.
* **New rule type** - register one function. The engine, the pipeline and the reports pick it up automatically.

Below: a generic `max_ratio` rule type, used to flag implausible taxi speeds (> 100 mph). It is passed
as an extra rule here; in production it would simply be added to the YAML.

In [13]:
@register_rule("max_ratio", category="invalid_value", required_params=("numerator", "denominator", "max"))
def rule_max_ratio(df, rule, ctx):
    num, den = F.col(rule["numerator"]), F.col(rule["denominator"])
    return df, [((den > 0) & (num / den > F.lit(rule["max"]) / F.lit(rule.get("denominator_scale", 1))), rule["name"])]


speed_rule = {"name": "implausible_speed", "type": "max_ratio", "numerator": "trip_distance",
              "denominator": "trip_duration_minutes", "max": 100, "denominator_scale": 60}   # 100 mph

dry = run_validated_ingestion("tripdata", TRIP_DIRTY, paths=SANDBOX, extra_rules=[speed_rule], dry_run=True)
display(dry["reasons"][["category", "reason", "action", "violations"]])

,category,reason,action,violations
0,duplicate,duplicate_in_batch,quarantine,30
1,duplicate,duplicate_existing,ignore,30
2,invalid_value,invalid_trip_distance,quarantine,20
3,missing_reference,missing_pickup_zone,quarantine,20
4,invalid_value,dropoff_before_pickup,quarantine,20
5,incomplete,incomplete:pickup_timestamp,quarantine,20
6,invalid_value,negative_fare,warn,20
7,incomplete,incomplete:event_timestamp,quarantine,20
8,invalid_value,invalid_vendor_id,quarantine,20
9,incomplete,incomplete:fare_amount,quarantine,20


## 10. Apply to the Task 1 update files (platform)

Runs the validated pipeline on the update files produced by `generate_incremental_updates.ipynb` and writes to the
**platform** tables. Re-running is safe: records already in Silver are ignored (`duplicate_existing`), so a second
run inserts nothing. If the update files have not been generated yet, this step is skipped.

In [14]:
UPDATE_FILES = {
    "tripdata": "data/raw/tripdata/tripdata_update_2024_04.parquet",
    "weather": "data/raw/weather/weather_update.csv",
    "air_quality": "data/raw/air_quality/air_quality_update.csv",
}

update_results = []
for ds, rel in UPDATE_FILES.items():
    path = os.path.join(PROJECT_ROOT, rel)
    if not os.path.exists(path):
        print(f"{ds}: {rel} not found - run notebooks/week3/generate_incremental_updates.ipynb first")
        continue
    update_results.append(run_validated_ingestion(ds, path, paths=PLATFORM))

if update_results:
    display(summarize(update_results))
    for r in update_results:
        print(f"\n{r['dataset_name']}: schema {r['schema_status']} {r['schema_changes']}")
        if not r["reasons"].empty:
            display(r["reasons"][["category", "reason", "action", "violations"]])

tripdata: data/raw/tripdata/tripdata_update_2024_04.parquet not found - run notebooks/week3/generate_incremental_updates.ipynb first
weather: data/raw/weather/weather_update.csv not found - run notebooks/week3/generate_incremental_updates.ipynb first
air_quality: data/raw/air_quality/air_quality_update.csv not found - run notebooks/week3/generate_incremental_updates.ipynb first


## 11. Discussion

**Generic rules (apply to every dataset, parameterised only by column names)**
* `unique` on the primary key and `not_in_table` against Silver - duplicate detection;
* `not_null` on primary-key, temporal and required columns - incomplete records;
* `cast_errors` - any value that cannot be converted to the common data model type (reuses Week 1 `change_datatype`);
* the schema checks - required columns, declared evolution, undeclared columns, type compatibility with Silver.
They follow from the common data model (every dataset has a primary key, an `event_timestamp`, typed columns
and a Silver table), so they are configured, never coded, per dataset.

**Dataset-specific rules (domain knowledge, expressed as configuration)**
* value domains: trip distance, passenger count, temperature, pressure, PM2.5, AQI 0–500, coordinates;
* code lists: vendor, payment type, rate code, EPA parameter 88101;
* cross-column logic: dropoff after pickup, trip ≤ 24 h (`expression`);
* referential integrity: taxi pickup/dropoff zones → `taxi_zone_lookup` (only trips reference another dataset);
* the *action* is a domain decision too: negative fares are refunds, so they only produce a warning; a weather
  hour delivered twice with different values keeps neither copy (`keep: none`), while identical taxi trips keep one.

**Adding rules without modifying the core framework**
* the engine (`ExtendedValidator`) and the pipeline only iterate over the configured rules and look their type up in
  `RULE_REGISTRY`; they contain no dataset or rule names;
* a new rule of an existing type is a YAML entry; `expression` accepts any Spark SQL predicate, which covers most
  new business rules without code;
* a new rule type is one function registered with `@register_rule` (section 9) - reporting, quarantine routing and
  monitoring work for it immediately;
* the configuration is checked when the validator is created (unknown type, missing parameter, duplicate name),
  so a mistake in a new rule fails before any data is processed.
